In [ ]:
## Code for 10-60 Data

# Associates linear and cyclized scaled counts for individual compounds via grouping and multindexing (lid, Common_Name (N-->C))
# Fits gaussian curves to every observable peak in the data
# Picks "best" peak and assigns retention time of centroid from gaussian fit
# Creates a new excel spreadsheet with calculated retention times

# Import packages as easy abbreviations
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
import os
%matplotlib inline
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.integrate import quad
from tqdm.auto import tqdm
from rdkit import Chem
from rdkit.Chem import Draw


# Define input spreadsheet. Define output folder for image files and name for output .xlsx file
excelsheet = "/Users/grant/Library/CloudStorage/GoogleDrive-brodohfasho@gmail.com/Shared drives/DEL Lipophilicity Paper/Data Spreadsheets/LDEL_ssPID_10-60_Master3.0.xlsx"
workbook = "Sheet1"
newdata = excelsheet.replace('3.0.xlsx','4.0.xlsx')


# Define peak picking parameters
# stddev_threshold = max standard deviation allowed for a fitted gaussian. Fit gaussians with standard deviations greater than this value are not plotted
# min_height_threshold_factor = the minimum peak height considered for gaussian fitting as a function of a percentage of the max y value (1 = only the max y peak is fit)
# fit_width = the width of the x-range around a peak to be used for fitting
# minimum_RT = excludes gaussian fitting for any peaks eluting before this RT (min)
stddev_threshold = 2
min_height_threshold_factor = 0.35
fit_width = 1.5
minimum_RT = 10


# Define gaussian equation
def gaussian(x, amplitude, mean, stddev):
        return amplitude * np.exp(-((x - mean) / (2 * stddev))**2)

# Create pandas dataframe from sequencing data
# Define indexes 
df = pd.read_excel(excelsheet, sheet_name=workbook)
df.set_index(["Common_Name (N-->C)", "lid"], inplace=True)

# Group by the first level of the MultiIndex
grouped_df = df.groupby(level=0)

# Create a dictionary for RT data collection
out_dict = {}

# Iterate over grouped data
for index, group_df in tqdm(grouped_df):
    out_dict[index] = list()

    for (lid, row) in group_df.iterrows():
        # Define new dataframe of delimited RT_count data
        chrom_df = pd.Series(row["all_datapoints"])

        # Split the comma-delimited values into a new dataframe (time, counts)
        chromvalues_df = chrom_df.str.split(",", expand=True)
        chromvalues_new = chromvalues_df.transpose()
        master_df = chromvalues_new[0].str.split("[:,;]", expand=True)

        # Filter out "None" values
        # Convert split delimited data from strings to numerical "float" values for plotting
        # Divide time points by 60 s to convert to minutes
        Time = [float(value) / 60 for value in master_df[0] if value is not None]
        Scaled_count = [float(value) for value in master_df[2] if value is not None]

        x = Time
        y = Scaled_count

        # Find peaks with a minimum height
        peaks, _ = find_peaks(y, height=max(y) * min_height_threshold_factor)

        gaussian_means = []

        # Fit and plot Gaussian for each peak
        for peak in peaks:
            # Define a range around the peak to fit the Gaussian
            # Determine the indices for fitting
            indices = [i for i, val in enumerate(x) if x[peak] - fit_width < val < x[peak] + fit_width]
            if not indices:
                continue

            fit_x = [x[i] for i in indices]
            fit_y = [y[i] for i in indices]

            # Ensure there are enough points to fit
            if len(fit_x) < 3:
                continue
            if x[peak] < minimum_RT:
                continue

            # Initial guess: amplitude, mean, stddev
            initial_guess = [max(fit_y), x[peak], np.std(fit_x)/2]
            bounds = ([0, x[peak] - fit_width, 0], [np.inf, x[peak] + fit_width, np.inf])

            try:
                popt, pcov = curve_fit(gaussian, fit_x, fit_y, p0=initial_guess, bounds=bounds, maxfev=1000)
                if popt[2] < stddev_threshold:
                    fit_x_values = np.linspace(min(fit_x)-0.5, max(fit_x)+0.5, 100)
                    w = gaussian(fit_x_values, *popt)
                    gaussian_means.append((popt[1], popt[0]))
                    
            except RuntimeError as e:
                # print(f"Could not fit a Gaussian to the peak at x={x[peak]}: {e}")
                pass

        best_mean = minimum_RT
        best_amplitude = 0
        for mean, amplitude in gaussian_means:
            if not amplitude > max(y) * min_height_threshold_factor:
                continue
            if mean > best_mean:
                best_mean = mean
                best_amplitude = amplitude

        if 'DEL-0044' not in lid:
            out_dict[index].append(best_mean)
        else:
            out_dict[index].insert(0, best_mean)

outDF = pd.DataFrame(out_dict).T
outDF.rename(columns={0:'Linear RT (min)',1:'Cyclized RT (min)'},inplace=True)
outDF.index.name = 'Common Name'

out_dict = {k:[x,y,y-x] for k, (x,y) in out_dict.items()}
Linear_RTs = list()
Cyclized_RTs = list()
deltaRT = list()
for ix0,ix1 in df.index:
    try:
        if 'DEL-0044' in ix1:
            Linear_RTs.append(out_dict[ix0][0])
            Cyclized_RTs.append(np.nan)
            deltaRT.append(out_dict[ix0][2])
        elif 'DEL-0045' in ix1:
            Cyclized_RTs.append(out_dict[ix0][1])
            Linear_RTs.append(np.nan)
            deltaRT.append(out_dict[ix0][2])
      
    except KeyError:
        print(f"DIDN'T FIND {ix0},{ix1}. setting to 'np.nan'")
        Linear_RTs.append(np.nan)
        Cyclized_RTs.append(np.nan)
        deltaRT.append(np.nan)
df['Linear RT (min)'] = Linear_RTs
df['Cyclized RT (min)'] = Cyclized_RTs
df['Delta RT (min) (Cyclized-Linear)'] = deltaRT

df.to_excel(newdata, merge_cells=False) 

  0%|          | 0/2 [00:00<?, ?it/s]

/var/folders/5m/wpxkkdws6gn53blbx7s7r2x40000gn/T/ipykernel_3282/561701028.py:101: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, pcov = curve_fit(gaussian, fit_x, fit_y, p0=initial_guess, bounds=bounds, maxfev=1000)
/var/folders/5m/wpxkkdws6gn53blbx7s7r2x40000gn/T/ipykernel_3282/561701028.py:101: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, pcov = curve_fit(gaussian, fit_x, fit_y, p0=initial_guess, bounds=bounds, maxfev=1000)


In [ ]:
## Code for 10-40 data - Sorts the data based on Time - Set Max Count Threshold Below (Unique to 10-40)

# Associates linear and cyclized scaled counts for individual compounds via grouping and multindexing (lid, Common_Name (N-->C))
# Fits gaussian curves to every observable peak in the data
# Picks "best" peak and assigns retention time of centroid from gaussian fit
# Creates a new excel spreadsheet with calculated retention times

# Import packages as easy abbreviations
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
import os
%matplotlib inline
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.integrate import quad
from tqdm.auto import tqdm
from rdkit import Chem
from rdkit.Chem import Draw


# Define input spreadsheet. Define output folder for image files and name for output .xlsx file
excelsheet = "D:/20240529_LDEL_10-40_AllGaussians/LDEL_ssPID_10-40_Master2.0.xlsx"
workbook = "Sheet1"
newdata = excelsheet.replace('2.0.xlsx','3.0.xlsx')


# Define peak picking parameters
# stddev_threshold = max standard deviation allowed for a fitted gaussian. Fit gaussians with standard deviations greater than this value are not plotted
# min_height_threshold_factor = the minimum peak height considered for gaussian fitting as a function of a percentage of the max y value (1 = only the max y peak is fit)
# fit_width = the width of the x-range around a peak to be used for fitting
# minimum_RT = excludes gaussian fitting for any peaks eluting before this RT (min)
stddev_threshold = 2
min_height_threshold_factor = 0.35
fit_width = 1.5
minimum_RT = 10


# Define gaussian equation
def gaussian(x, amplitude, mean, stddev):
        return amplitude * np.exp(-((x - mean) / (2 * stddev))**2)

# Create pandas dataframe from sequencing data
# Define indexes 
df = pd.read_excel(excelsheet, sheet_name=workbook)
df.set_index(["Common_Name (N-->C)", "lid"], inplace=True)

# Group by the first level of the MultiIndex
grouped_df = df.groupby(level=0)

# Create a dictionary for RT data collection
out_dict = {}

# Iterate over grouped data
for index, group_df in tqdm(grouped_df):
    out_dict[index] = list()

    for (lid, row) in group_df.iterrows():
        # Check if the "max_count" value is less than 100
        if row["max_count"] < 30:
            continue  # Skip the Gaussian fitting for this entry

        # Define new dataframe of delimited RT_count data
        chrom_df = pd.Series(row["all_datapoints"])

        # Split the comma-delimited values into a new dataframe (time, counts)
        chromvalues_df = chrom_df.str.split(",", expand=True)
        chromvalues_new = chromvalues_df.transpose()
        master_df = chromvalues_new[0].str.split("[:,;]", expand=True)

        # Create a DataFrame with the time and count values
        data = pd.DataFrame({'Time': master_df[0].astype(float) / 60, 'Count': master_df[2].astype(float)})

        # Sort the DataFrame based on the 'Time' column
        data = data.sort_values('Time')

        # Extract the sorted time and count values
        x = data['Time'].tolist()
        y = data['Count'].tolist()

        # Find peaks with a minimum height
        peaks, _ = find_peaks(y, height=max(y) * min_height_threshold_factor)

        gaussian_means = []

        # Fit and plot Gaussian for each peak
        for peak in peaks:
            # Define a range around the peak to fit the Gaussian
            # Determine the indices for fitting
            indices = [i for i, val in enumerate(x) if x[peak] - fit_width < val < x[peak] + fit_width]
            if not indices:
                continue

            fit_x = [x[i] for i in indices]
            fit_y = [y[i] for i in indices]

            # Ensure there are enough points to fit
            if len(fit_x) < 3:
                continue
            if x[peak] < minimum_RT:
                continue

            # Initial guess: amplitude, mean, stddev
            initial_guess = [max(fit_y), x[peak], np.std(fit_x)/2]
            bounds = ([0, x[peak] - fit_width, 0], [np.inf, x[peak] + fit_width, np.inf])

            try:
                popt, pcov = curve_fit(gaussian, fit_x, fit_y, p0=initial_guess, bounds=bounds, maxfev=1000)
                if popt[2] < stddev_threshold:
                    gaussian_means.append((popt[1], popt[0]))
                    
            except RuntimeError as e:
                # print(f"Could not fit a Gaussian to the peak at x={x[peak]}: {e}")
                pass

        best_mean = minimum_RT
        best_amplitude = 0
        for mean, amplitude in gaussian_means:
            if mean > best_mean:
                best_mean = mean
                best_amplitude = amplitude

        if 'DEL-0044' not in lid:
            out_dict[index].append(best_mean)
        else:
            out_dict[index].insert(0, best_mean)

# Find the maximum length among all arrays in out_dict
max_length = max(len(arr) for arr in out_dict.values())

# Pad shorter arrays with np.nan to make all arrays have the same length
for key in out_dict:
    out_dict[key] = out_dict[key] + [np.nan] * (max_length - len(out_dict[key]))

outDF = pd.DataFrame(out_dict).T
outDF.rename(columns={0:'Linear RT (min)',1:'Cyclized RT (min)'},inplace=True)
outDF.index.name = 'Common Name'

out_dict = {k:[x,y,y-x] for k, (x,y) in out_dict.items()}
Linear_RTs = []
Cyclized_RTs = []
deltaRT = []

for ix0, ix1 in df.index:
    if ix0 in out_dict:
        if 'DEL-0044' in ix1:
            Linear_RTs.append(out_dict[ix0][0])
            Cyclized_RTs.append(np.nan)
            deltaRT.append(out_dict[ix0][2])
        elif 'DEL-0045' in ix1:
            Cyclized_RTs.append(out_dict[ix0][1])
            Linear_RTs.append(np.nan)
            deltaRT.append(out_dict[ix0][2])
    else:
        Linear_RTs.append(np.nan)
        Cyclized_RTs.append(np.nan)
        deltaRT.append(np.nan)

df['Linear RT (min)'] = Linear_RTs
df['Cyclized RT (min)'] = Cyclized_RTs
df['Delta RT (min) (Cyclized-Linear)'] = deltaRT

df.to_excel(newdata, merge_cells=False)